# 04 - Statistical Analysis
**Project:** Hopper C3 E-Commerce | **Dataset:** UK Online Retail


In [ ]:
import pandas as pd; import numpy as np; import matplotlib.pyplot as plt; import seaborn as sns
from scipy import stats; from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split; from sklearn.metrics import r2_score, mean_absolute_error
import statsmodels.api as sm; import warnings; warnings.filterwarnings('ignore')
df = pd.read_csv('../cleaned_data.csv', parse_dates=['InvoiceDate'])
df['IsWeekend'] = df['Day'].isin(['Saturday','Sunday']).astype(int)
df['IsUK'] = (df['Country']=='United Kingdom').astype(int)
print(f'Loaded {len(df):,} rows'); df.head(3)

## 2. Correlation Analysis

In [ ]:
corr=df[['Quantity','UnitPrice','Revenue','Month']].corr()
fig,ax=plt.subplots(figsize=(8,6))
sns.heatmap(corr,annot=True,fmt='.3f',cmap='RdYlGn',center=0,mask=np.triu(np.ones_like(corr,dtype=bool)),ax=ax)
ax.set_title('Pearson Correlation Matrix'); plt.tight_layout(); plt.show()
print(corr['Revenue'].drop('Revenue').sort_values(ascending=False).to_string())

## 3. Linear Regression

In [ ]:
r=df[(df.Quantity>0)&(df.Revenue>0)].copy()
r['lQ']=np.log1p(r.Quantity); r['lR']=np.log1p(r.Revenue)
Xtr,Xte,ytr,yte=train_test_split(r[['lQ']],r.lR,test_size=0.2,random_state=42)
mod=LinearRegression().fit(Xtr,ytr); yp=mod.predict(Xte)
print(f'R2={r2_score(yte,yp):.4f}  MAE={mean_absolute_error(yte,yp):.4f}')

## 4. OLS Regression

In [ ]:
r['lUP']=np.log1p(r.UnitPrice)
X=sm.add_constant(r[['lQ','lUP','Month','IsWeekend','IsUK']])
ols=sm.OLS(r.lR,X).fit(); print(ols.summary())

## 5. Hypothesis Testing

In [ ]:
iv=df.groupby(['InvoiceNo','IsWeekend']).Revenue.sum().reset_index()
t,p=stats.ttest_ind(iv[iv.IsWeekend==0].Revenue,iv[iv.IsWeekend==1].Revenue,equal_var=False)
print(f'Weekday vs Weekend t-test: p={p:.4f}')
ig=df.groupby(['InvoiceNo','IsUK']).Revenue.sum().reset_index()
u,pm=stats.mannwhitneyu(ig[ig.IsUK==1].Revenue,ig[ig.IsUK==0].Revenue,alternative='two-sided')
print(f'UK vs International MW-U: p={pm:.4f}')
im=df.groupby(['InvoiceNo','Month']).Revenue.sum().reset_index()
f,pa=stats.f_oneway(*[g.Revenue.values for _,g in im.groupby('Month')])
print(f'Monthly ANOVA: p={pa:.4f}')

## 6. Seasonality

In [ ]:
mo=df.groupby(df.InvoiceDate.dt.to_period('M')).Revenue.sum().reset_index()
mo['InvoiceDate']=mo.InvoiceDate.astype(str); mo['Roll3']=mo.Revenue.rolling(3,center=True).mean()
fig,ax=plt.subplots(figsize=(14,5))
ax.bar(mo.InvoiceDate,mo.Revenue,color='lightsteelblue',label='Monthly')
ax.plot(mo.InvoiceDate,mo.Roll3,color='red',lw=2,marker='o',ms=4,label='3M Avg')
ax.set_title('Monthly Revenue - 3M Rolling Avg')
plt.xticks(rotation=45,ha='right'); ax.legend(); plt.tight_layout(); plt.show()

## 7. Summary

In [ ]:
print(f'OLS R2={ols.rsquared:.4f} | Weekday p={p:.4f} | UK p={pm:.4f} | ANOVA p={pa:.4f}')